## 1. Setup and Dependencies

In [ ]:
# --- Clean install + repo-consistent imports ---

# 1) Remove pip openwakeword to avoid mixing with cloned repo code
%pip uninstall -y openwakeword

# 2) Install system + python deps (without openwakeword)
!apt-get -qq update
!apt-get -qq install -y espeak-ng
%pip install -q datasets soundfile transformers accelerate bitsandbytes huggingface_hub piper-sample-generator tqdm pronouncing espeak-phonemizer torch-audiomentations speechbrain audiomentations mutagen torchinfo torchmetrics acoustics scipy numpy pyyaml deep-phonemizer==0.0.19 scikit-learn requests

# 3) Move to /content
%cd /content

import os
import sys
import shutil
import subprocess
import time
from pathlib import Path

work = Path("/content")

def git_clone_repo(repo_url: str, target_dir: Path, retries: int = 3):
    if target_dir.exists():
        shutil.rmtree(target_dir)

    for i in range(retries):
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", repo_url, str(target_dir)],
                check=True, text=True, capture_output=True
            )
            print(f"Cloned {repo_url} -> {target_dir}")
            return
        except subprocess.CalledProcessError as e:
            print(f"[{i+1}/{retries}] clone failed: {e.stderr}")
            time.sleep(3)
    raise RuntimeError(f"Failed to clone {repo_url}")

# 4) Clone repos
openwakeword_dir = work / "openWakeWord"
piper_generator_dir = work / "piper-sample-generator"
git_clone_repo("https://github.com/dscripka/openWakeWord.git", openwakeword_dir)
git_clone_repo("https://github.com/dscripka/piper-sample-generator.git", piper_generator_dir)

# 4b) Download the exact Piper model expected by generate_samples.py
# generate_samples defaults to models/en-us-libritts-high.pt
subprocess.run([
    "wget", "-q", "-O", str(piper_generator_dir / "models" / "en-us-libritts-high.pt"),
    "https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt",
], check=True)
print("Piper model ready:", piper_generator_dir / "models" / "en-us-libritts-high.pt")

# 5) Install openWakeWord FROM THE CLONED REPO (editable)
#    This is the key fix for your ImportError mismatch.
%pip install -q -e /content/openWakeWord

# 6) Ensure repo path wins in this kernel too
if str(openwakeword_dir) not in sys.path:
    sys.path.insert(0, str(openwakeword_dir))
if str(piper_generator_dir) not in sys.path:
    sys.path.insert(0, str(piper_generator_dir))

# 7) Create dirs expected by your workflow
(work / "audioset_16k").mkdir(parents=True, exist_ok=True)
(work / "fma").mkdir(parents=True, exist_ok=True)

# 8) Verify imports are from /content/openWakeWord
#    and proactively validate train-time dependencies.
import openwakeword, openwakeword.data
import dp
import torch, torchaudio, torchinfo, torchmetrics
import numpy, scipy, yaml
import pronouncing, audiomentations, torch_audiomentations
import speechbrain, mutagen, acoustics

# 9) Ensure required OpenWakeWord ONNX resource models exist in editable clone.
from openwakeword.utils import download_file
resources_models_dir = openwakeword_dir / "openwakeword" / "resources" / "models"
resources_models_dir.mkdir(parents=True, exist_ok=True)

feature_model_urls = [
    spec["download_url"] for spec in openwakeword.FEATURE_MODELS.values()
]
for url in feature_model_urls:
    tflite_name = url.split("/")[-1]
    onnx_name = tflite_name.replace(".tflite", ".onnx")
    if not (resources_models_dir / tflite_name).exists():
        print(f"Downloading missing feature model: {tflite_name}")
        download_file(url, str(resources_models_dir))
    if not (resources_models_dir / onnx_name).exists():
        print(f"Downloading missing feature model: {onnx_name}")
        download_file(url.replace(".tflite", ".onnx"), str(resources_models_dir))

required_onnx = [
    resources_models_dir / "melspectrogram.onnx",
    resources_models_dir / "embedding_model.onnx",
]
for p in required_onnx:
    if not p.exists():
        raise FileNotFoundError(f"Missing required openWakeWord model file: {p}")

print("openwakeword:", openwakeword.__file__)
print("openwakeword.data:", openwakeword.data.__file__)
print("deep-phonemizer(dp):", getattr(dp, "__file__", "loaded"))
print("openwakeword resource models: OK")
print("train deps preflight: OK")
print("cwd:", os.getcwd())

Found existing installation: openwakeword 0.6.0
Uninstalling openwakeword-0.6.0:
  Successfully uninstalled openwakeword-0.6.0
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/content
Cloned https://github.com/dscripka/openWakeWord.git -> /content/openWakeWord
Cloned https://github.com/dscripka/piper-sample-generator.git -> /content/piper-sample-generator
Piper model ready: /content/piper-sample-generator/models/en-us-libritts-high.pt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done
openwakeword: /content/openWakeWord/openwakeword/__init__.py
openwakeword.data: /content/openWakeWord/openwakeword/data.py
cwd: /content


## 2. Download Precomputed Feature Files

Download the required precomputed feature files from Hugging Face for efficient training.

In [2]:
from huggingface_hub import hf_hub_download

# Download precomputed feature files from the public dataset repo
DATASET_REPO = "davidscripka/openwakeword_features"

hf_hub_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    local_dir="/content/",
)
hf_hub_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    filename="validation_set_features.npy",
    local_dir="/content/",
)

print("Precomputed feature files downloaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: 401 Client Error. (Request ID: Root=1-6a408a2a-3989510873a955ea1c863e8b;4d952321-154f-4dca-b487-d574c0e65da5)

Repository Not Found for url: https://huggingface.co/openwakeword/openwakeword-features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a privat

Precomputed feature files downloaded.


## 3. Collect Background Audio (AudioSet Only)

We will collect background audio clips from the AudioSet dataset. The `rudraml/fma` dataset has been excluded as per requirements due to streaming issues.

In [3]:
import datasets
import soundfile as sf
from tqdm import tqdm
from pathlib import Path

# Self-contained paths so this cell works even if earlier cells were skipped/reordered.
work = Path("/content")
out_a = work / "audioset_16k"
out_f = work / "fma"
out_a.mkdir(parents=True, exist_ok=True)
out_f.mkdir(parents=True, exist_ok=True)

print("Downloading AudioSet background clips...")

ds_a = datasets.load_dataset("agkphysics/AudioSet", split="train", streaming=True)
ds_a = iter(ds_a.cast_column("audio", datasets.Audio(sampling_rate=16000)))

# Download 400 AudioSet clips
for i in tqdm(range(400), desc="Downloading AudioSet clips"):
    try:
        row = next(ds_a)
        sf.write(out_a / f"audioset_{i:04d}.wav", row["audio"]["array"], 16000)
    except Exception as e:
        print(f"Skipping AudioSet clip {i} due to error: {e}")

# Print status for FMA, as it's not being downloaded
print("Skipping FMA dataset download due to known issues with 'datasets' library streaming for this dataset.")
print("FMA data will be checked for existence in write_config and included if available if downloaded by other means.")

print("Background audio collection complete.")

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Skipping FMA dataset download due to known issues with 'datasets' library streaming for this dataset.
FMA data will be checked for existence in write_config and included if available if downloaded by other means.
Background audio collection complete.


## 4. Generate Synthetic Speech for "Reachy Wake up"

Use `piper-sample-generator` to create synthetic audio clips for the target wake word phrase.

In [ ]:
import sys
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi
from huggingface_hub.utils import HfHubHTTPError # Corrected import
import subprocess
import os
import random

# 'work' is Path('/content') from previous cells

# Create a directory for raw synthetic speech (before augmentation)
raw_synthetic_output_dir = Path("/content") / "raw_synthetic_speech"
raw_synthetic_output_dir.mkdir(exist_ok=True)

# Remove stale files so listening checks reflect current generation only.
for old_wav in raw_synthetic_output_dir.glob("*.wav"):
    old_wav.unlink()

# Define wakeword pronunciation variants for better TTS coverage.
# Piper voices can pronounce "Reachy" differently; these variants reduce bias.
wakeword_phrase_variants = [
    "Reachy wake up",
    "Ree-chee wake up",
    "Reechee wake up",
    "Reachi wake up",
]

# Guard against accidental literal "text " prefix in TTS phrases.
wakeword_phrase_variants = [
    p.strip()[5:].strip() if p.strip().lower().startswith("text ") else p.strip()
    for p in wakeword_phrase_variants
]

# Define paths for a Piper model and its config
# We'll store downloaded models inside the cloned piper-sample-generator repo's models dir
piper_model_base_dir = Path("/content") / "piper-sample-generator" / "models"
piper_model_base_dir.mkdir(parents=True, exist_ok=True)

# --- Dynamically find multiple Piper TTS models --- START
piper_voice_paths = []

print("Attempting to find available Piper TTS models...")
try:
    api = HfApi()
    repo_files = api.list_repo_files("rhasspy/piper-voices")

    # Prioritize a diverse set of English voices.
    preferred_models = [
        "en_US-amy-medium.onnx",
        "en_US-ryan-high.onnx",
        "en_US-lessac-medium.onnx",
        "en_US-libritts-high.onnx",
        "en_GB-alan-medium.onnx",
        "en_GB-northern_english_male-medium.onnx",
    ]

    found_models = []
    for f in repo_files:
        if f.startswith("en/") and f.endswith(".onnx"):
            model_name_only = Path(f).name
            config_name_only = model_name_only.replace(".onnx", ".onnx.json")
            config_full_path = str(Path(f).parent / config_name_only)
            if config_full_path in repo_files:
                found_models.append((f, config_full_path))

    selected_models = []

    # Pick preferred models first (if present)
    for pref_model_filename in preferred_models:
        for model_full_path, config_full_path in found_models:
            if Path(model_full_path).name == pref_model_filename:
                selected_models.append((model_full_path, config_full_path))
                break

    # Backfill with any remaining English models
    max_voices = 4
    if len(selected_models) < max_voices:
        for model_full_path, config_full_path in found_models:
            if (model_full_path, config_full_path) not in selected_models:
                selected_models.append((model_full_path, config_full_path))
                if len(selected_models) >= max_voices:
                    break

    selected_models = selected_models[:max_voices]

    if not selected_models:
        raise ValueError("No suitable English Piper ONNX models found in rhasspy/piper-voices.")

    print("Selected Piper voices:")
    for model_rel, _ in selected_models:
        print(f"  - {model_rel}")

    for model_rel, config_rel in selected_models:
        model_local = piper_model_base_dir / model_rel
        config_local = piper_model_base_dir / config_rel
        model_local.parent.mkdir(parents=True, exist_ok=True)
        config_local.parent.mkdir(parents=True, exist_ok=True)

        hf_hub_download(repo_id="rhasspy/piper-voices", filename=model_rel, local_dir=piper_model_base_dir)
        hf_hub_download(repo_id="rhasspy/piper-voices", filename=config_rel, local_dir=piper_model_base_dir)

        piper_voice_paths.append((model_local, config_local))

    print(f"Downloaded {len(piper_voice_paths)} Piper voices.")
except HfHubHTTPError as e:
    print(f"Error accessing Hugging Face Hub: {e}")
    raise RuntimeError("Failed to access Hugging Face Hub. Cannot proceed with speech generation.")
except Exception as e:
    print(f"Error finding or downloading Piper TTS models: {e}")
    raise RuntimeError("Failed to download Piper TTS models. Cannot proceed with speech generation.")

# --- Dynamically find multiple Piper TTS models --- END

# Path to the piper executable (should be in PATH after pip install piper-tts, a dependency)
piper_executable = "piper" # Assumes 'piper' is in system PATH. For Colab it typically is /usr/local/bin/piper

print(f"Generating raw synthetic speech for variants {wakeword_phrase_variants} to {raw_synthetic_output_dir}...")

# Generate clips per selected voice for better speaker diversity.
samples_per_voice = 8
sample_idx = 0

for voice_idx, (model_path, config_path) in enumerate(piper_voice_paths):
    print(f"\nVoice {voice_idx + 1}/{len(piper_voice_paths)}: {model_path.name}")
    for _ in range(samples_per_voice):
        output_wav_file = raw_synthetic_output_dir / f"raw_sample_{sample_idx:03d}.wav"
        sample_idx += 1

        phrase = random.choice(wakeword_phrase_variants)
        if phrase.lower().startswith("text "):
            raise ValueError(f"Unexpected phrase with 'text ' prefix: {phrase}")

        command = [
            piper_executable,
            "--model", str(model_path),
            "--output_file", str(output_wav_file),
            "--config", str(config_path),
        ]
        try:
            subprocess.run(
                command,
                input=phrase + "\n",
                check=True,
                capture_output=True,
                text=True,
                env={**os.environ, "PATH": f"/usr/local/bin:{os.environ.get('PATH', '')}"},
            )
            print(f"Generated {output_wav_file.name} <- '{phrase}' (stdin)")
        except FileNotFoundError:
            print("Error: 'piper' executable not found. Ensure piper-tts is installed and in PATH.")
            raise
        except subprocess.CalledProcessError as e:
            print(f"Error generating {output_wav_file.name}: {e}")
            if e.stderr:
                print(f"Stderr: {e.stderr}")

if not any(raw_synthetic_output_dir.glob("*.wav")):
    raise RuntimeError("No raw synthetic speech samples were generated. Cannot proceed.")
else:
    print(f"Raw synthetic speech generation complete. Total clips: {sample_idx}")

Attempting to find an available Piper TTS model...
Found and selected Piper model: en/en_US/amy/medium/en_US-amy-medium.onnx
Attempting to download the selected Piper TTS model...


en/en_US/amy/medium/en_US-amy-medium.onn(…):   0%|          | 0.00/63.2M [00:00<?, ?B/s]

en_US-amy-medium.onnx.json:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

Piper TTS model and config downloaded.
Generating raw synthetic speech for 'Reachy Wake up' to /content/raw_synthetic_speech...
Running: piper --model /content/piper-sample-generator/models/en/en_US/amy/medium/en_US-amy-medium.onnx --output_file /content/raw_synthetic_speech/raw_sample_00.wav --text Reachy Wake up --config /content/piper-sample-generator/models/en/en_US/amy/medium/en_US-amy-medium.onnx.json
Generated /content/raw_synthetic_speech/raw_sample_00.wav
Running: piper --model /content/piper-sample-generator/models/en/en_US/amy/medium/en_US-amy-medium.onnx --output_file /content/raw_synthetic_speech/raw_sample_01.wav --text Reachy Wake up --config /content/piper-sample-generator/models/en/en_US/amy/medium/en_US-amy-medium.onnx.json
Generated /content/raw_synthetic_speech/raw_sample_01.wav
Running: piper --model /content/piper-sample-generator/models/en/en_US/amy/medium/en_US-amy-medium.onnx --output_file /content/raw_synthetic_speech/raw_sample_02.wav --text Reachy Wake up --

## 4.5. Augment Synthetic Speech

Augment the raw synthetic speech clips to create a more robust training dataset.

In [5]:
from pathlib import Path

# 'work' is Path('/content') from previous cells
# 'raw_synthetic_output_dir' holds the raw generated samples
# 'synthetic_output_dir' is where the augmented samples should go (as expected by YAML)

# Ensure the synthetic_output_dir exists
synthetic_output_dir = Path("/content") / "synthetic_speech"
synthetic_output_dir.mkdir(exist_ok=True)

print(f"Augmenting synthetic speech from {raw_synthetic_output_dir} to {synthetic_output_dir}...")
# Corrected: piper_sample_generator.augment expects positional arguments without explicit flags
!python3 -m piper_sample_generator.augment {raw_synthetic_output_dir} {synthetic_output_dir} --sample-rate 16000

if not any(synthetic_output_dir.glob("*.wav")):
    raise RuntimeError("No augmented synthetic speech samples were generated. Cannot proceed.")
else:
    print("Synthetic speech augmentation complete.")

Augmenting synthetic speech from /content/raw_synthetic_speech to /content/synthetic_speech...
/usr/local/lib/python3.12/dist-packages/piper_sample_generator/augment.py:3: DeprecationWarning: 'audioop' is deprecated and slated for removal in Python 3.13
  import audioop
/usr/local/lib/python3.12/dist-packages/audiomentations/core/audio_loading_utils.py:37: UserWarning: /usr/local/lib/python3.12/dist-packages/piper_sample_generator/impulses/Fat Bass.wav had to be resampled from 44100 hz to 22050 hz. This hurt execution time.
  warnings.warn(
/content/synthetic_speech/raw_sample_09.wav
/usr/local/lib/python3.12/dist-packages/audiomentations/core/audio_loading_utils.py:37: UserWarning: /usr/local/lib/python3.12/dist-packages/piper_sample_generator/impulses/Symphonic.wav had to be resampled from 44100 hz to 22050 hz. This hurt execution time.
  warnings.warn(
/content/synthetic_speech/raw_sample_03.wav
/content/synthetic_speech/raw_sample_07.wav
/content/synthetic_speech/raw_sample_01.wav


## 5. Write openWakeWord Training YAML Configuration

Create the `reachy_wake_up.yaml` configuration file for training the custom model.

In [ ]:
import yaml
from pathlib import Path

# Self-contained paths so config generation doesn't depend on prior variable state.
work = Path("/content")
out_a = work / "audioset_16k"
out_f = work / "fma"
out_a.mkdir(parents=True, exist_ok=True)
out_f.mkdir(parents=True, exist_ok=True)

config_path = work / "reachy_wake_up.yaml"

background_paths = [str(out_a)]
fma_path = work / "fma"
if fma_path.exists() and any(fma_path.glob("*.wav")):
    background_paths.append(str(fma_path))

config_data = {
    # required identity
    "target_phrase": [
        "reachy wake up",
        "ree-chee wake up",
        "reechee wake up",
        "reachi wake up",
    ],
    "model_name": "reachy_wake_up",

    # optional explicit negatives (keep empty to auto-generate phoneme-overlap negatives)
    "custom_negative_phrases": [],

    # synthetic sample generation
    "n_samples": 30000,
    "n_samples_val": 2000,
    "tts_batch_size": 50,

    # paths
    "output_dir": str(work / "reachy_wake_up"),
    "background_paths": background_paths,
    "background_paths_duplication_rate": [1] * len(background_paths),
    "rir_paths": [],

    # required precomputed feature datasets
    "false_positive_validation_data_path": str(work / "validation_set_features.npy"),
    "feature_data_files": {
        "ACAV100M_sample": str(work / "openwakeword_features_ACAV100M_2000_hrs_16bit.npy")
    },

    # training batch composition
    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50,
    },

    # model/training parameters expected by train.py
    "model_type": "dnn",
    "layer_size": 32,
    "steps": 50000,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,

    # kept for compatibility with notebook comments/older variants
    "target_accuracy": 0.7,
    "target_recall": 0.5,

    # REQUIRED by current train.py for clip generation
    "piper_sample_generator_path": str(work / "piper-sample-generator"),

    # generation/augmentation knobs
    "augmentation_batch_size": 16,
    "augmentation_rounds": 1,
}

print(f"Using background_paths: {background_paths}")

with open(config_path, "w") as f:
    yaml.safe_dump(config_data, f, sort_keys=False)

print(f"Configuration file saved to {config_path}")

Using background_paths: ['/content/audioset_16k']
Configuration file saved to /content/reachy_wake_up.yaml


## 6. Run Training Stages

Execute the three training stages: `generate_clips`, `augment_clips`, and `train_model`.

In [7]:
import subprocess
import sys
import os
from pathlib import Path

TRAIN_DIR = Path("/content/openWakeWord/openwakeword")
TRAIN_PY = TRAIN_DIR / "train.py"
CFG = Path("/content/reachy_wake_up.yaml")
PIPER_GEN_DIR = Path("/content/piper-sample-generator")

assert TRAIN_DIR.exists(), f"Missing trainer directory: {TRAIN_DIR}"
assert TRAIN_PY.exists(), f"Missing trainer script: {TRAIN_PY}"
assert CFG.exists(), f"Missing config: {CFG}"

def run_stage(flag: str, extra_args=None):
    import time
    from collections import deque

    # Fix 1: PyTorch 2.6+ security bypass for Piper models.
    # Also patch torchaudio>=2.9 compatibility for torch_audiomentations
    # by providing a fallback torchaudio.info() backed by soundfile.
    allowlist_logic = (
        "import torch, types; import piper_train.vits.models; "
        "torch.serialization.add_safe_globals([piper_train.vits.models.SynthesizerTrn, piper_train.vits.models.TextEncoder]); "
        "import torchaudio; import soundfile as sf; "
        "hasattr(torchaudio, 'info') or setattr(torchaudio, 'info', "
        "(lambda path, format=None, buffer_size=4096, backend=None: "
        "types.SimpleNamespace(sample_rate=sf.info(path).samplerate, num_frames=sf.info(path).frames, num_channels=sf.info(path).channels, bits_per_sample=0, encoding=getattr(sf.info(path), 'subtype', 'UNKNOWN')))); "
    )

    # Fix 2: Ensure piper-sample-generator is in the PYTHONPATH for the subprocess
    env = os.environ.copy()
    python_path = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = f"{PIPER_GEN_DIR}:{TRAIN_DIR.parent}:{python_path}"

    # Fix 3: Colab PyTorch 2.6+ defaults torch.load(weights_only=True),
    # which breaks piper-sample-generator checkpoints.
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

    argv = ["train.py", "--training_config", str(CFG), flag]
    if extra_args:
        argv.extend(extra_args)

    cmd = [
        sys.executable, "-u", "-c",
        f"{allowlist_logic} import sys; sys.argv={argv!r}; exec(open('{TRAIN_PY}').read())"
    ]

    print("\n$", f"Running {flag} with live logs (this can take a while)...", flush=True)

    p = subprocess.Popen(
        cmd,
        cwd=str(TRAIN_DIR),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        bufsize=1,
    )

    tail = deque(maxlen=120)
    last_output = time.time()

    while True:
        line = p.stdout.readline()
        if line:
            print(line, end="", flush=True)
            tail.append(line.rstrip("\n"))
            last_output = time.time()
            continue

        if p.poll() is not None:
            break

        if time.time() - last_output >= 60:
            print(f"[still running {flag}... no new log lines yet]", flush=True)
            last_output = time.time()

        time.sleep(0.2)

    rc = p.wait()
    if rc != 0:
        print("\n--- Combined output tail (last lines) ---")
        print("\n".join(tail) if tail else "(none)")
        raise RuntimeError(f"{flag} failed with exit code {rc}")

print("Python:", sys.executable)
print("Stage 1: generate clips")
run_stage("--generate_clips")

print("Stage 2: augment clips")
# Force rebuild to avoid stale/partial feature files from previous attempts.
run_stage("--augment_clips", ["--overwrite"])

print("Stage 3: train model")
run_stage("--train_model")

print("All training stages complete.")

Python: /usr/bin/python3
Stage 1: generate clips

$ Running --generate_clips with PyTorch security bypass and updated PYTHONPATH...

--- STDOUT (tail) ---
(none)

--- STDERR (tail) ---
INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en-us-libritts-high.pt
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "<string>", line 676, in <module>
  File "/content/piper-sample-generator/generate_samples.py", line 74, in generate_samples
    model = torch.load(model_path)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1578, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if

RuntimeError: --generate_clips failed with exit code 1

In [ ]:
import os
import subprocess
import sys

# Get the site-packages path to ensure pronouncing is found
try:
    result = subprocess.run([sys.executable, '-c', 'import site; print(site.getsitepackages()[0])'], capture_output=True, text=True, check=True)
    site_packages_path = result.stdout.strip()
    print(f"Site packages path: {site_packages_path}")
except Exception as e:
    print(f"Could not determine site-packages path: {e}")
    site_packages_path = None

# Set PYTHONPATH for the subprocess calls to openwakeword.train
# This ensures modules installed via pip are discoverable by the new python process
custom_env = os.environ.copy()
if site_packages_path:
    current_pythonpath = custom_env.get('PYTHONPATH', '')
    # Add the site-packages path if not already present
    if site_packages_path not in current_pythonpath.split(os.pathsep):
        custom_env['PYTHONPATH'] = f"{site_packages_path}{os.pathsep}{current_pythonpath}" if current_pythonpath else site_packages_path
    print(f"PYTHONPATH set for openwakeword.train: {custom_env['PYTHONPATH']}")
else:
    print("Warning: PYTHONPATH not explicitly set for openwakeword.train due to missing site-packages path.")

# Stage 1: Generate clips
print("Running --generate_clips stage...")
try:
    result = subprocess.run([
        sys.executable,
        "-m", "openwakeword.train",
        "--config_file", "/content/reachy_wake_up.yaml",
        "--generate_clips",
        "--output_dir", "/content/"
    ], check=True, capture_output=True, text=True, env=custom_env)
    print("Generate clips stage complete.")
    if result.stdout: print(f"Stdout from openwakeword.train: {result.stdout}")
    if result.stderr: print(f"Stderr from openwakeword.train: {result.stderr}")
except subprocess.CalledProcessError as e:
    print(f"Error during --generate_clips stage: {e}")
    if e.stdout: print(f"Stdout from openwakeword.train: {e.stdout}")
    if e.stderr: print(f"Stderr from openwakeword.train: {e.stderr}")
    raise # Re-raise the error so the user knows it failed

# Stage 2: Augment clips
print("Running --augment_clips stage...")
try:
    result = subprocess.run([
        sys.executable,
        "-m", "openwakeword.train",
        "--config_file", "/content/reachy_wake_up.yaml",
        "--augment_clips",
        "--output_dir", "/content/"
    ], check=True, capture_output=True, text=True, env=custom_env)
    print("Augment clips stage complete.")
    if result.stdout: print(f"Stdout from openwakeword.train: {result.stdout}")
    if result.stderr: print(f"Stderr from openwakeword.train: {result.stderr}")
except subprocess.CalledProcessError as e:
    print(f"Error during --augment_clips stage: {e}")
    if e.stdout: print(f"Stdout from openwakeword.train: {e.stdout}")
    if e.stderr: print(f"Stderr from openwakeword.train: {e.stderr}")
    raise # Re-raise the error so the user knows it failed

# Stage 3: Train model
print("Running --train_model stage...")
try:
    result = subprocess.run([
        sys.executable,
        "-m", "openwakeword.train",
        "--config_file", "/content/reachy_wake_up.yaml",
        "--train_model",
        "--output_dir", "/content/"
    ], check=True, capture_output=True, text=True, env=custom_env)
    print("Train model stage complete.")
    if result.stdout: print(f"Stdout from openwakeword.train: {result.stdout}")
    if result.stderr: print(f"Stderr from openwakeword.train: {result.stderr}")
except subprocess.CalledProcessError as e:
    print(f"Error during --train_model stage: {e}")
    if e.stdout: print(f"Stdout from openwakeword.train: {e.stdout}")
    if e.stderr: print(f"Stderr from openwakeword.train: {e.stderr}")
    raise # Re-raise the error so the user knows it failed

print("All training stages complete.")

## 7. Verify ONNX Output

Confirm that the `reachy_wake_up.onnx` model file has been successfully created.

In [ ]:
output_onnx_path = work / "reachy_wake_up.onnx"
if output_onnx_path.exists():
    print(f"Successfully created ONNX model at: {output_onnx_path}")
else:
    print(f"Error: ONNX model not found at {output_onnx_path}")

## 8. SCP Command to Copy Model

Here is the `scp` command to copy the trained ONNX model to your specified destination.

In [ ]:
scp_command = "scp /content/reachy_wake_up.onnx pollen@reachy-mini.local:/home/pollen/clawbody/scripts/scheduler/models/"
print(f"To copy the model to your Reachy Mini, run the following command from your local machine (not in Colab):")
print(f"```bash\n{scp_command}\n```")